In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/detection/crowd.mp4


## 1. One-Stage Подход: YOLOv11

In [12]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.1 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.4.5.8-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.2.1.3-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.5.147-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.6.1.9-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.3.1.170-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_nvjitlink_cu12-12.4.127

In [13]:
from ultralytics import YOLO
import cv2
import numpy as np
from tqdm import tqdm

# Загрузка модели
model = YOLO('yolo11m.pt')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [18]:

# Параметры для overhead detection
conf_threshold = 0.3 # 0.3 - 0.55, уверенность модели в классе
iou_threshold = 0.45   # 0.4 - 0.45
# Если уменьшить (например, до 0.3-0.4), 
# то NMS будет жестче отсеивать перекрывающиеся боксы — помогает избежать многократных детекций одного объекта, 
# но рискует удалить близкие объекты в плотных толпах. Если увеличить (до 0.5-0.6), 
# допускается больше перекрытия, что может привести к коллизиям.


def process_video_yolo(video_path, output_path):
    # Открываем исходное видео
    cap = cv2.VideoCapture(video_path)

    # Получаем параметры видео: fps, ширина, высота
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # Настраиваем writer для сохранения результата в mp4 (кодек mp4v)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    frame_count = 0               # Счетчик кадров
    total_persons = []            # Список для подсчета людей на каждом кадре
    
    # Обрабатываем видео по кадрам
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # инференс  YOLO
        # Получаем только людей (class 0); conf/iou_threshold задаются вне функции
        results = model(
            frame,
            conf=conf_threshold, 
            iou=iou_threshold, 
            classes=[0],          # Находить только людей
            verbose=False
        )
        
        # Получаем аннотированный кадр с выделенными людьми (boxes)
        annotated = results[0].plot()
        
        # Подсчитываем количество найденных объектов (людей) на кадре
        boxes = results[0].boxes
        num_persons = len(boxes)
        total_persons.append(num_persons)
        

        # Сохраняем итоговый кадр в выходное видео
        out.write(annotated)
        frame_count += 1
    
    # Освобождаем ресурсы
    cap.release()
    out.release()
    
    # Итоговая статистика — среднее, мин, макс людей на кадр
    return {
        'avg_persons': np.mean(total_persons),
        'max_persons': np.max(total_persons),
        'min_persons': np.min(total_persons)
    }

# Запуск
stats = process_video_yolo('/kaggle/input/detection/crowd.mp4', '/kaggle/working/output_yolo11_30.mp4')
print(f"Статистика YOLOv11: {stats}")


Статистика YOLOv11: {'avg_persons': 11.539007092198581, 'max_persons': 15, 'min_persons': 7}


## 2. Two-Stage Подход: Mask R-CNN

In [3]:
import torch
import torchvision
from torchvision.models.detection.mask_rcnn import maskrcnn_resnet50_fpn
import torchvision.transforms as T
import cv2
import numpy as np

In [4]:

# Загрузка предобученной модели
model = maskrcnn_resnet50_fpn(pretrained=True)
model.eval()
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MaskRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth" to /root/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth
100%|██████████| 170M/170M [00:00<00:00, 218MB/s] 


MaskRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(in

In [5]:
# Предобработка кадра
def preprocess(frame):
    transform = T.Compose([T.ToTensor()])
    return transform(frame).to(device)

# Детекция
def detect(frame):
    img = preprocess(frame)
    with torch.no_grad():
        preds = model([img])[0]
    return preds


In [10]:
# Пример обработки видео
cap = cv2.VideoCapture('/kaggle/input/detection/crowd.mp4')
# Получаем параметры видео
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# Создаем объект VideoWriter для сохранения обработанного видео
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('/kaggle/working/output_maskrcnn80.mp4', fourcc, fps, (width, height))

while True:
    ret, frame = cap.read()
    if not ret:  # Если достигнут конец видео — выйти из цикла
        break
    preds = detect(frame)
    # Извлекаем предсказанные боксы, метки классов и confidence-оценки
    boxes = preds['boxes']
    labels = preds['labels']  # label = 1 - person
    scores = preds['scores']
    
    for box, label, score in zip(boxes, labels, scores):
        if label == 1 and score > 0.50:  # можно пробовать score > 0.3 / 0.5 / 0.65 в зависимости от цели
            x1, y1, x2, y2 = map(int, box)
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
    out.write(frame)

cap.release()
out.release()